In [35]:

from datetime import datetime
from pyspark.sql.types import *
import uuid

# === CONFIGURATION - Change for each notebook ===
NOTEBOOK_NAME = "ntk_Sil2Gld_DimSites V2"        # ← Change this for each notebook
PIPELINE_NAME = "pipeline_test"     # ← Change this for each pipeline
ACTIVITY_TYPE = "DataTransformation"     # ← DataExtract/DataTransform/DataLoad/DataValidation
SOURCE_PATH = "abfss://Silver/Dim_Site" # ← Change source path
TARGET_PATH = "abfss://Gold/Dim_Site" # ← Change target path

def log_etl_activity(status, start_time=None, error=None, **metrics):
    """Log ETL activity to pipeline table"""
    current_time = datetime.now()
    
    # Get next LogID
    try:
        log_id = spark.sql("SELECT COALESCE(MAX(LogID), 0) + 1 as id FROM etl_gold_pipeline_log").collect()[0]['id']
    except:
        log_id = 1
    
    if status == "STARTED":
        data = [(
            log_id, PIPELINE_NAME, f"run_{current_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            current_time, None, None, "RUNNING", None, 
            SOURCE_PATH, TARGET_PATH, None, None, None, None, 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
        start_time = current_time
        
    else:
        duration = int((current_time - start_time).total_seconds()) if start_time else None
        data = [(
            log_id, PIPELINE_NAME, f"run_{start_time.strftime('%Y%m%d_%H%M%S')}", 
            NOTEBOOK_NAME, ACTIVITY_TYPE, NOTEBOOK_NAME, 1, 
            start_time, current_time, duration, status, 
            str(error) if error else None, SOURCE_PATH, TARGET_PATH,
            metrics.get('rows_read'), metrics.get('rows_written'), 
            metrics.get('file_count'), metrics.get('bytes_processed'), 
            current_time, "system", str(uuid.uuid4())[:8]
        )]
    
    # Schema for etl_gold_pipeline_log table
    schema = StructType([
        StructField("LogID", LongType()), StructField("PipelineName", StringType()),
        StructField("RunID", StringType()), StructField("ActivityName", StringType()),
        StructField("ActivityType", StringType()), StructField("NotebookName", StringType()),
        StructField("Sequence", IntegerType()), StructField("StartTime", TimestampType()),
        StructField("EndTime", TimestampType()), StructField("DurationSeconds", IntegerType()),
        StructField("Status", StringType()), StructField("ErrorMessage", StringType()),
        StructField("SourcePath", StringType()), StructField("TargetPath", StringType()),
        StructField("RowsRead", LongType()), StructField("RowsWritten", LongType()),
        StructField("FileCountProcessed", IntegerType()), StructField("BytesProcessed", LongType()),
        StructField("InsertedOn", TimestampType()), StructField("InsertedBy", StringType()),
        StructField("CorrelationID", StringType())
    ])
    
    # Save to table
    spark.createDataFrame(data, schema).write.mode("append").saveAsTable("etl_gold_pipeline_log")
    
    # Print status
    if status == "STARTED":
        print(f"🚀 Starting {NOTEBOOK_NAME}")
    elif status == "SUCCESS":
        duration_text = f" ({duration}s)" if duration else ""
        print(f"✅ {NOTEBOOK_NAME} completed successfully{duration_text}")
    else:
        print(f"❌ {NOTEBOOK_NAME} failed")
    
    return current_time if status == "STARTED" else None

# Start logging
print(f"🔧 Initializing {NOTEBOOK_NAME}...")
start_time = log_etl_activity("STARTED")

# Initialize variables for tracking metrics
rows_read = 0
rows_written = 0
file_count = 0
bytes_processed = 0

# print(f"📊 Ready to process data from: {SOURCE_PATH}")
# print(f"🎯 Target location: {TARGET_PATH}")

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 37, Finished, Available, Finished)

🔧 Initializing ntk_Sil2Gld_DimSites V2...
🚀 Starting ntk_Sil2Gld_DimSites V2


In [36]:
# Section --- Importing modules and Defining Variable for standard usage source / target.
from pyspark.sql import SparkSession
from datetime import datetime, timedelta
import os

# Base source path (up to Files level)
base_source_path = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files"
# Base target path  
base_target_path  = "abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files"

today = datetime.now()  
from datetime import datetime, timedelta
# today = today - timedelta(days=1)

year = today.strftime("%Y")
month = today.strftime("%m")
day = today.strftime("%d")  

# Build dynamic source folder structure
sourcefolder_structure = f"Silver_layer/Reporting/{year}/{month}/{day}"   
# Complete source path
complete_source_path = f"{base_source_path}/{sourcefolder_structure}"
# Filename
source_filename = "Dim_Site.parquet"

# Build dynamic target folder structure
targetfolder_structure = f"/PreGold_Reporting"   
# Complete target path
complete_target_path = f"{base_target_path}/{targetfolder_structure}"
# Filename
target_filename = "Dim_Site.parquet"
print(f"Variables created and session started.")


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 38, Finished, Available, Finished)

Variables created and session started.


In [37]:
# # Initialize Spark session
# spark = SparkSession.builder.appName("SilverToGold_Sites").getOrCreate()

# Full Source  file path
full_source_path = f"{complete_source_path}/{source_filename}"

# Full Target  file path
full_target_path = f"{complete_target_path}/{target_filename}"

print(f"Source: {full_source_path}")
print(f"Target: {full_target_path}")


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 39, Finished, Available, Finished)

Source: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Site.parquet
Target: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Site.parquet


In [38]:
# Final full paths
# full_source_path = f"{complete_source_path}/{source_filename}"
# full_target_path = f"{complete_target_path}/{target_filename}"

## 🔹 3. **Initialize Spark Session**
spark = SparkSession.builder.appName("SilverToGold_Sites").getOrCreate()

## 🔹 4. **Read Source and Target Data**
source_df = spark.read.parquet(full_source_path)

# try:
target_df = spark.read.parquet(full_target_path)
# except:
#     target_df = spark.createDataFrame([], source_df.schema)

print(f"Reading source file completed {full_source_path}")
print(f"Reading target file completed {full_target_path}")
source_df.show()
target_df.show()

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 40, Finished, Available, Finished)

Reading source file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Silver_lakehouse.Lakehouse/Files/Silver_layer/Reporting/2025/10/15/Dim_Site.parquet
Reading target file completed abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Site.parquet
+-------------+--------------------+--------------------+-------------------+------------+-----------+-------------+---------+--------------------+--------------------+-------------------+----------+-------------+-------------+--------------+------------------+-------------+-------------+---------+----------+---------+--------+--------+--------+--------------------+--------------------+------------------+---------------+--------------------+-----------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------+-------------------+-------+----------

### Caches

In [39]:
# Cache for performance
target_df.cache()
print(target_df.count())

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 41, Finished, Available, Finished)

1


In [40]:
## 🔹 5. **Align Source Columns to Target Columns**
### 🛠 Function to Rename and Cast Source Columns
from pyspark.sql.functions import col
from pyspark.sql.types import *

def align_to_target_schema_only(source_df, target_df, source_columns, target_columns):
    target_schema = {field.name: field.dataType for field in target_df.schema.fields}
    aligned_df = source_df.select(source_columns)
    target_df = target_df.select(target_columns)
    mapped_target_cols = set()

    for src_col, tgt_col in zip(source_columns, target_columns):
        if src_col in aligned_df.columns and src_col != tgt_col:
            aligned_df = aligned_df.withColumnRenamed(src_col, tgt_col)
            mapped_target_cols.add(tgt_col)

    for tgt_col in mapped_target_cols:
        if tgt_col in aligned_df.columns:
            target_type = target_schema.get(tgt_col, StringType())
            aligned_df = aligned_df.withColumn(tgt_col, col(tgt_col).cast(target_type))

    return aligned_df

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 42, Finished, Available, Finished)

In [41]:
## 🔹 6. **Prepare & Align Source and Target Schemas**
source_cols = ["SiteKey", "SiteId",	"WebId", "Title", "Site_Url", "Owner", "OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description","SensitivityLabel","SharingCapability","ConditionalAccessPolicy","Classification","DatacenterLocation","CreatedBy","Created","LastModifiedBy","LastModified"]
target_cols = ["SiteKey", "SiteID", "WebId", "SiteName", "SiteURL", "Owner","OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description","SensitivityLabel","SharingCapability","ConditionalAccessPolicy","Classification","DCLocation","CreatedBy","CreatedDate","ModifiedBy","ModifiedDate"]

upd_source_df = source_df.select(source_cols)
upd_target_df = target_df.select(target_cols)
upd_source_df = align_to_target_schema_only(upd_source_df, upd_target_df, source_cols, target_cols)
upd_source_df.show()


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 43, Finished, Available, Finished)

+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------+--------+--------+------------+----------------+--------------------+-----------------------+--------------+-------------+--------------------+--------------------+----------+--------------------+
|             SiteKey|              SiteID|               WebId|     SiteName|             SiteURL|               Owner|           OwnerName|          OwnerEmail|Language|LocaleId|TimeZone| Description|SensitivityLabel|   SharingCapability|ConditionalAccessPolicy|Classification|   DCLocation|           CreatedBy|         CreatedDate|ModifiedBy|        ModifiedDate|
+--------------------+--------------------+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------+--------+--------+------------+----------------+--------------------+-----------------------+

In [42]:
from pyspark.sql.functions import col, current_timestamp

def detect_records2_inserts(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (source.key = target.key)
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Find records in source not in target
    inserts_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "left_anti")
        .withColumn("ModifiedDate", current_timestamp())
    )
    
    insert_count = inserts_df.count()
    print(f"🆕 INSERT: {insert_count} records found")
    
    return inserts_df.select(sourcekey)

def detect_records2updates(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition
    join_condition = col(f"s.{sourcekey}") == col(f"t.{targetkey}")
    
    # Get non-key columns for comparison
    key_column = sourcekey
    non_key_cols = [c for c in source_df.columns if c != key_column]
    
    # Build dynamic "any column differs" filter
    diff_condition = None
    for c in non_key_cols:
        cond = col(f"s.{c}") != col(f"t.{c}")
        diff_condition = cond if diff_condition is None else (diff_condition | cond)
    
    # print(f"Condition:{diff_condition}")
    # Find records that exist in both but have differences
    updates_df = (
        source_df.alias("s")
        .join(target_df.alias("t"), join_condition, "inner")
        .filter(diff_condition)  # Only records with differences
        .select(
            col("s.*"),  # Take updated values from source
            current_timestamp().alias("ModifiedDate")
        )
    )
    
    update_count = updates_df.count()
    print(f"✏️ UPDATE: {update_count} records found")
    
    return updates_df.select(sourcekey)

def detect_records2_deletes(source_df, target_df, sourcekey, targetkey):
    
    # Cache for performance
    source_df.cache()
    target_df.cache()
    
    # Build join condition (target.key = source.key) 
    join_condition = col(f"t.{targetkey}") == col(f"s.{sourcekey}")
    
    # Find records in target not in source
    deletes_df = (
        target_df.alias("t")
        .join(source_df.alias("s"), join_condition, "left_anti")
        .withColumn("LastModified", current_timestamp())
    )
    
    delete_count = deletes_df.count()
    print(f"🗑️ DELETE: {delete_count} records found")
    
    return deletes_df.select(targetkey)


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 44, Finished, Available, Finished)

In [51]:
# # Code to check new records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# # Method 1a: Create single new record using Row 
# # *****PLEASE NOTE, THIS IS MANUAL ENTRY FOR TESTING AND NEED TO BE REMOVED ONCEE PRODUCTIONIZE.
# upd_source_df.show(50)
# new_record = spark.createDataFrame([
#     Row(
#         SiteKey = "SK001",
#         SiteID = "101",
#         WebId = "W001",
#         SiteName = "AlphaSite",
#         SiteURL = "https://alpha.example.com",
#         Owner = "user123",
#         OwnerName = "John Doe",
#         OwnerEmail = "john.doe@example.com",
#         Language = "en",
#         LocaleId = "1033",
#         TimeZone = "Pacific Standard Time",
#         Description = "Main site for Alpha project",
#     )
# ])

# upd_source_df = upd_source_df.union(new_record)
# upd_source_df.show(2)
# # **** MANUAL CODES INSERT *****

new_result_df = detect_records2_inserts(upd_source_df, upd_target_df,"SiteKey","SiteKey")
new_result_df.show(1)
# print(f"target before:{target_df.count()}")

# # Since its redone in below code removed from here 10/15 *****
# column_mapping = dict(zip(target_cols, target_cols))
# new_records_df = (
#     new_result_df.alias("n")
#     .join(upd_source_df.alias("s"), col("n.SiteKey") == col("s.SiteKey"), "inner")
#     .select(*[col(f"s.{source_cols}").alias(target_cols) for source_cols, target_cols in column_mapping.items()])
# )

# print(f"target before:{new_records.count()}")

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 53, Finished, Available, Finished)

🆕 INSERT: 0 records found
+-------+
|SiteKey|
+-------+
+-------+



In [47]:
new_records_df.show()

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 49, Finished, Available, Finished)

+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+
|SiteKey|SiteID|WebId|SiteName|SiteURL|Owner|OwnerName|OwnerEmail|Language|LocaleId|TimeZone|Description|SensitivityLabel|SharingCapability|ConditionalAccessPolicy|Classification|DCLocation|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+



In [48]:
# # Code to check removed record from source and remove from target table dataframe *****
from pyspark.sql import Row
from pyspark.sql.functions import lit, current_timestamp, col
from pyspark.sql.types import *

# Delete Record in users 
# upd_source_df.filter(col("OwnerName") == "sureshgupta@peopletech.com").show()
# # Remove specific user by OwnerName 
# # ***** THIS IS TEST RECORD AND NEED TO BE REMOVED ON PRODUCTION LOAD.
# # upd_source_df = upd_source_df.filter(col("OwnerName") != "sureshgupta")

# # Perform removal of deleted records - in-memory merge
delete_result_df = detect_records2_deletes(upd_source_df, upd_target_df,"SiteKey","SiteKey")
delete_result_df.show(2)

print(f"Source deleted record has been locted")

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 50, Finished, Available, Finished)

🗑️ DELETE: 0 records found
+-------+
|SiteKey|
+-------+
+-------+

Source deleted record has been locted


In [49]:
# # Code to check updated records and add to dataframe *****

from pyspark.sql import Row
from pyspark.sql.functions import * 
# import lit, current_timestamp, col,when, upper, trim
from pyspark.sql.types import *

# # Update Department for all users in "Software Engineering" 
# # THIS IS SAMPLE COD TO MODIFY RECORD FOR TESTING. SHOULD BE REMOVE ONCE PRODUCTION
# upd_source_df.filter(col("SiteId") == "5b37303a-58ba-4f8e-b7f3-48f07e230df8").show()
# upd_source_df = upd_source_df.withColumn(
#     "OwnerName",
#     when(col("SiteId") == "5b37303a-58ba-4f8e-b7f3-48f07e230df8", "peopletech")
#     .otherwise(col("OwnerName"))
# )
# upd_source_df.filter(col("SiteId") == "5b37303a-58ba-4f8e-b7f3-48f07e230df8").show()

# source_cols = ["SiteKey", "SiteId",	"WebId", "Title", "Site_Url", "Owner", "OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description", "CreatedByUser", "Created", "LastModifiedByUser", "LastModified", "SnapShotDate"]
# target_cols = ["SiteKey", "SiteID", "WebId", "SiteName", "SiteURL", "Owner","OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description",  "CreatedBy", "CreatedDate", "ModifiedBy", "ModifiedDate", "SnapshotDate"]

upd_result_df = detect_records2updates(upd_source_df, upd_target_df,"SiteKey","SiteKey")
upd_result_df.show()



StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 51, Finished, Available, Finished)

✏️ UPDATE: 0 records found
+-------+
|SiteKey|
+-------+
+-------+



In [50]:

################# # # Code to update target dataframe with updated / inserrted / deleted records *************8
from pyspark.sql.functions import lit, current_date
from pyspark.sql.types import StringType

# new_records_df.show()
# upd_result_df.show()
final_result_Key = (
    new_records_df.select(col("SiteKey").alias("SiteKey"))
    .union(upd_result_df.select(col("SiteKey").alias("SiteKey")))
)
final_result_Key.show()

# Get updated records with specified columns only
# source_cols = ["SiteKey", "SiteId",	"WebId", "Title", "Site_Url", "Owner", "OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description"]
# target_cols = ["SiteKey", "SiteID", "WebId", "SiteName", "SiteURL", "Owner","OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description"]

# updated_add_records = (
#     final_result_Key.alias("n")
#     .join(source_df.alias("s"), col("n.SiteKey") == col("s.SiteKey") , "inner")
#     .select(*[f"s.{col}" for col in source_cols])
# )

updated_add_records = (
    final_result_Key.alias("n")
    .join(upd_source_df.alias("s"), col("n.SiteKey") == col("s.SiteKey") , "inner")
    .select(*[f"s.{col}" for col in target_cols])
)

# updated_add_records = target_df.withColumn("IsDeleted", lit(0))
updated_add_records = updated_add_records.withColumn("IsDeleted", lit(0))
updated_add_records.show(1)

# # # #  # # updated_add_records = align_to_target_schema_only(updated_add_records, updated_delrecords, source_cols, target_cols

# updated_add_records = align_to_target_schema_only(updated_add_records, updated_delrecords, source_cols, target_cols)
# updated_add_records.show(2)


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 52, Finished, Available, Finished)

+-------+
|SiteKey|
+-------+
+-------+

+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
|SiteKey|SiteID|WebId|SiteName|SiteURL|Owner|OwnerName|OwnerEmail|Language|LocaleId|TimeZone|Description|SensitivityLabel|SharingCapability|ConditionalAccessPolicy|Classification|DCLocation|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+------

In [34]:
# # code to creeate delete recor dataframe and update isdeleted to 1 and other system values.
# target_cols = ["SiteKey", "SiteID", "WebId", "SiteName", "SiteURL", "Owner","OwnerName", "OwnerEmail", "Language", "LocaleId", "TimeZone", "Description"]

updated_delrecords = (
    delete_result_df.alias("n")
    .join(upd_target_df.alias("s"), col("n.SiteKey") == col("s.SiteKey") , "inner")
    .select(*[f"s.{col}" for col in target_cols])
)
updated_delrecords = updated_delrecords.withColumn("IsDeleted", lit(1))
# updated_delrecords = updated_delrecords \
#     .withColumn("Type", lit("Site")) \
#     .withColumn("SensitivityLabel",lit(None).cast(StringType())) \
#     .withColumn("Type", lit("Site").cast(StringType())) \
#     .withColumn("SensitivityLabel",lit(None).cast(StringType())) \
#     .withColumn("Classification",lit(None).cast(StringType())) \
#     .withColumn("DCLocation",lit(None).cast(StringType())) \
#     .withColumn("SiteStatus",lit("Active").cast(StringType())) \
#     .withColumn("SharingCapability",lit(None).cast(StringType())) \
#     .withColumn("ConditionalAccessPolicy",lit(None).cast(StringType())) \
#     .withColumn("IsDeleted", lit(1)) \
#     .withColumn("SnapshotDate", current_date())
updated_delrecords.show(1)
# updated_add_records.show(2)

final_target_rec = updated_add_records.unionByName(updated_delrecords, allowMissingColumns=True)
final_target_rec.show()

# final_records = (
#      updated_delrecords
#     .union(updated_add_records)
# )
# final_records = updated_add_records.unionByName(updated_delrecords, allowMissingColumns=True)
# final_records = final_records.select(*target_df.columns)

# # # Remove records from target that exist in updadddeeel, then add updated records
# final_target_rec = (
#     target_df.join(final_records.select("SiteKey"), "SiteKey", "left_anti")  # Remove existing
#     .union(final_records)  # Add updated records
# )
# final_target_rec.show()


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 36, Finished, Available, Finished)

+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
|SiteKey|SiteID|WebId|SiteName|SiteURL|Owner|OwnerName|OwnerEmail|Language|LocaleId|TimeZone|Description|SensitivityLabel|SharingCapability|ConditionalAccessPolicy|Classification|DCLocation|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+

+-------+--

In [40]:
 # ===== WRITE PROCESS =====
try:
    # Write to Gold layer (PreGold_Reporting)
    print(f"Writing to Gold layer: {full_target_path}")

    full_target_path_new = full_target_path
    ####.rsplit('.', 1)[0]}_{datetime.now():%Y%m%d}.parquet"
    final_target_rec.coalesce(1) \
        .write \
        .mode("overwrite") \
        .option("compression", "snappy") \
        .parquet(full_target_path_new)

    # final_target_rec.write.mode("overwrite").option("compression", "snappy") \
    # .parquet(f"{full_target_path_new}")

    # final_target.write.mode("overwrite").option("compression", "snappy") \
    # .parquet(f"{full_target_path.rsplit('.',1)[0]}_{datetime.now():%Y%m%d}.parquet")
    print(f"Successfully written to: {full_target_path_new}")
    
    # Verify the written file
    df_verify = spark.read.parquet(full_target_path_new)
    print(f"Verification - Target record count: {df_verify.count()}")
    
except Exception as e:
    print(f"Error in processing: {str(e)}")
    print("Please check:")
    print(f"1. Source file exists: {full_source_path}")
    print(f"2. Target path is accessible: {base_target_path}")

print("Process completed!")


StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 20, Finished, Available, Finished)

Writing to Gold layer: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Site.parquet


Successfully written to: abfss://BoeingProjectAudit@onelake.dfs.fabric.microsoft.com/Boeing_ForGold.Lakehouse/Files//PreGold_Reporting/Dim_Site.parquet


Verification - Target record count: 1
Process completed!


In [43]:
# final_target_rec.show(5)
# final_target_rec.printSchema()

StatementMeta(, d244b8dc-ce04-49a9-8037-584f1538ac09, 21, Finished, Available, Finished)

+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
|SiteKey|SiteID|WebId|SiteName|SiteURL|Owner|OwnerName|OwnerEmail|Language|LocaleId|TimeZone|Description|SensitivityLabel|SharingCapability|ConditionalAccessPolicy|Classification|DCLocation|CreatedBy|CreatedDate|ModifiedBy|ModifiedDate|IsDeleted|
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+
+-------+------+-----+--------+-------+-----+---------+----------+--------+--------+--------+-----------+----------------+-----------------+-----------------------+--------------+----------+---------+-----------+----------+------------+---------+

root
 |-- S

In [42]:

# Ensure processing_successful is defined before this block
try:
    print(f"🔄 Starting ETL processing for {NOTEBOOK_NAME}...")

    # Your ETL logic here
    # Example:
    # source_df = spark.read.format("delta").load(SOURCE_PATH)
    # transformed_df = source_df.filter("status = 'active'")
    # transformed_df.write.format("delta").mode("overwrite").save(TARGET_PATH)

    # Simulated metrics
    rows_read = source_df.count()
    rows_written = final_target_rec.count()
    # file_count = len(dbutils.fs.ls(SOURCE_PATH))
    bytes_processed = 524288000  # ~500MB

    processing_successful = True

except Exception as e:
    error_details = e
    processing_successful = False

# Complete the logging based on processing results
if processing_successful:
    # Log successful completion with metrics
    log_etl_activity("SUCCESS", start_time, 
                     rows_read=rows_read, 
                     rows_written=rows_written,
                    #  file_count=file_count,
                     bytes_processed=bytes_processed)
    
    print(f"🎉 {NOTEBOOK_NAME} pipeline completed successfully!")
    print(f"📊 Final metrics:")
    print(f"   ✅ Status: SUCCESS")
    print(f"   📖 Total rows processed: {rows_read:,} → {rows_written:,}")
    print(f"   🔄 Data throughput: {bytes_processed/(1024**2):.1f} MB")
    
    # Optional: Show recent logs for this notebook
    print(f"\n📋 Recent runs for {NOTEBOOK_NAME}:")
    spark.sql(f"""
        SELECT LogID, Status, StartTime, EndTime, DurationSeconds, RowsRead, RowsWritten
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=True)
    
else:
    # Log failure
    log_etl_activity("FAILED", start_time, error_details)
    
    print(f"💥 {NOTEBOOK_NAME} pipeline failed!")
    print(f"❌ Error: {str(error_details)}")
    
    # Optional: Show error analysis
    print(f"\n🔍 Recent failures for debugging:")
    spark.sql(f"""
        SELECT LogID, StartTime, ErrorMessage, DurationSeconds
        FROM etl_silver_pipeline_log 
        WHERE NotebookName = '{NOTEBOOK_NAME}' AND Status = 'FAILED'
        ORDER BY StartTime DESC 
        LIMIT 3
    """).show(truncate=False)
    
    # Re-raise the exception to fail the notebook
    raise error_details

# Cleanup variables
print(f"\n🧹 Cleaning up variables...")
del rows_read, rows_written, bytes_processed

print(f"✨ {NOTEBOOK_NAME} logging completed!")


StatementMeta(, b0060a36-1722-4000-b43b-bfea4112a380, 31, Finished, Available, Finished)

🔄 Starting ETL processing for ntk_Sil2Gld_DimSites V2...


✅ ntk_Sil2Gld_DimSites V2 completed successfully (1261s)
🎉 ntk_Sil2Gld_DimSites V2 pipeline completed successfully!
📊 Final metrics:
   ✅ Status: SUCCESS
   📖 Total rows processed: 1 → 1
   🔄 Data throughput: 500.0 MB

📋 Recent runs for ntk_Sil2Gld_DimSites V2:


+-----+------+---------+-------+---------------+--------+-----------+
|LogID|Status|StartTime|EndTime|DurationSeconds|RowsRead|RowsWritten|
+-----+------+---------+-------+---------------+--------+-----------+
+-----+------+---------+-------+---------------+--------+-----------+


🧹 Cleaning up variables...
✨ ntk_Sil2Gld_DimSites V2 logging completed!
